In [1]:
# ============================================================
# 04_phase2_clustering_annotation_clean.ipynb
# Phase 2 — Clustering and Cell Type Annotation
# GSE114725 (Azizi et al. 2018) and GSE176078 (Wu et al. 2021)
# ============================================================

# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
from pathlib import Path
from scipy.sparse import issparse

sc.settings.verbosity = 1

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase2_clustering_v2"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Ready")

Ready


In [2]:
# ----------------------------
# Cell 2 — Load Phase 1 v2 objects
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase1_v2.h5ad")
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase1_v2.h5ad")

print("GSE114725:", adata1.n_obs, "cells x", adata1.n_vars, "genes")
print("GSE176078:", adata2.n_obs, "cells x", adata2.n_vars, "genes")
print("\nGSE114725 obs:", list(adata1.obs.columns))
print("GSE176078 obs:", list(adata2.obs.columns))

GSE114725: 44662 cells x 2000 genes
GSE176078: 91425 cells x 2000 genes

GSE114725 obs: ['patient', 'tissue', 'replicate', 'cluster', 'n_genes_by_counts', 'total_counts', 'doublet_score', 'predicted_doublet']
GSE176078 obs: ['Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset', 'n_genes_by_counts', 'total_counts', 'doublet_score', 'predicted_doublet']


In [3]:
# ----------------------------
# Cell 3 — Leiden clustering at multiple resolutions
# Neighbours built on X_pca_harmony (Harmony-corrected PCA)
# ----------------------------
sc.pp.neighbors(adata1, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30)
sc.pp.neighbors(adata2, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30)

for res in [0.2, 0.4, 0.6, 0.8, 1.0]:
    sc.tl.leiden(adata1, resolution=res, key_added=f"leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False)
    sc.tl.leiden(adata2, resolution=res, key_added=f"leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False)
    print(f"Resolution {res}: GSE114725={adata1.obs[f'leiden_{res}'].nunique()} "
          f"clusters, GSE176078={adata2.obs[f'leiden_{res}'].nunique()} clusters")

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Resolution 0.2: GSE114725=2 clusters, GSE176078=16 clusters
Resolution 0.4: GSE114725=2 clusters, GSE176078=22 clusters
Resolution 0.6: GSE114725=6 clusters, GSE176078=26 clusters
Resolution 0.8: GSE114725=6 clusters, GSE176078=31 clusters
Resolution 1.0: GSE114725=9 clusters, GSE176078=33 clusters


In [4]:
# ----------------------------
# Cell 4 — Resolution comparison UMAPs
# Selected: resolution 0.6 for both datasets
# GSE114725: 6 biologically distinct clusters, all supported by canonical markers
# GSE176078: 26 clusters covering full TME diversity
# Lower resolutions merge biologically distinct populations
# Higher resolutions create ambiguous clusters not supported by canonical markers
# ----------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, res in zip(axes, [0.4, 0.6, 0.8]):
    sc.pl.umap(adata1, color=f"leiden_{res}", title=f"GSE114725 res={res}",
               legend_loc="on data", legend_fontsize=8, ax=ax, show=False)
plt.suptitle("GSE114725 — Resolution Comparison", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_resolution_comparison.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, res in zip(axes, [0.4, 0.6, 0.8]):
    sc.pl.umap(adata2, color=f"leiden_{res}", title=f"GSE176078 res={res}",
               legend_loc="on data", legend_fontsize=6, ax=ax, show=False)
plt.suptitle("GSE176078 — Resolution Comparison", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE176078_resolution_comparison.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Resolution comparison figures saved")

Resolution comparison figures saved


In [4]:
# ----------------------------
# Cell 4 — Resolution comparison UMAPs
# ----------------------------
fig, axes = plt.subplots(1, 5, figsize=(30, 5))
for i, res in enumerate(["leiden_0.2", "leiden_0.4", "leiden_0.6", "leiden_0.8", "leiden_1.0"]):
    sc.pl.umap(adata1, color=res, title=f"GSE114725 {res}",
               legend_loc="on data", legend_fontsize=8, ax=axes[i], show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_resolution_comparison.png", dpi=300, bbox_inches="tight")
plt.close()

fig, axes = plt.subplots(1, 5, figsize=(30, 5))
for i, res in enumerate(["leiden_0.2", "leiden_0.4", "leiden_0.6", "leiden_0.8", "leiden_1.0"]):
    sc.pl.umap(adata2, color=res, title=f"GSE176078 {res}",
               legend_loc="on data", legend_fontsize=8, ax=axes[i], show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE176078_resolution_comparison.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved — open figures to inspect")

Saved — open figures to inspect


In [5]:
# ----------------------------
# Cell 5 — Marker genes per cluster
# Run on leiden_0.6 for both datasets
# Wilcoxon rank-sum test — appropriate for identifying cluster markers
# Note: V(D)J genes removed in Phase 1 so no contamination of marker lists
# ----------------------------
sc.tl.rank_genes_groups(adata1, groupby="leiden_0.6", method="wilcoxon",
                        key_added="rank_genes_leiden_0.6")
sc.tl.rank_genes_groups(adata2, groupby="leiden_0.6", method="wilcoxon",
                        key_added="rank_genes_leiden_0.6")

markers1 = sc.get.rank_genes_groups_df(adata1, group=None, key="rank_genes_leiden_0.6")
markers2 = sc.get.rank_genes_groups_df(adata2, group=None, key="rank_genes_leiden_0.6")

markers1.to_csv(RESULTS_DIR / "GSE114725_markers_v2.csv", index=False)
markers2.to_csv(RESULTS_DIR / "GSE176078_markers_v2.csv", index=False)
markers1.groupby("group").head(10).to_csv(
    RESULTS_DIR / "GSE114725_top10_markers_v2.csv", index=False)
markers2.groupby("group").head(10).to_csv(
    RESULTS_DIR / "GSE176078_top10_markers_v2.csv", index=False)

print("=== GSE114725 Top 5 markers per cluster ===")
for cl in sorted(markers1["group"].unique(), key=lambda x: int(x)):
    genes = markers1[markers1["group"] == cl].head(5)["names"].tolist()
    print(f"Cluster {cl}: {', '.join(genes)}")

print("\n=== GSE176078 Top 5 markers per cluster ===")
for cl in sorted(markers2["group"].unique(), key=lambda x: int(x)):
    genes = markers2[markers2["group"] == cl].head(5)["names"].tolist()
    print(f"Cluster {cl}: {', '.join(genes)}")

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\tools\_rank_genes_groups.py:458: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\tools\_rank_genes_groups.py:460: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\tools\_rank_genes_groups.py:463: PerformanceWarning: DataFrame is high

=== GSE114725 Top 5 markers per cluster ===
Cluster 0: RPLP2, RPS29, RPL13, RPS27, RPS12
Cluster 1: DUSP1, FOS, ZFP36, FOSB, RGS1
Cluster 2: NKG7, B2M, CCL5, GNLY, PRF1
Cluster 3: CD74, HLA-DRA, FTH1, HLA-DPA1, FTL
Cluster 4: CD79A, CD74, HLA-DRA, CD37, HLA-DPB1
Cluster 5: CST3, FTL, PSAP, FTH1, TYROBP

=== GSE176078 Top 5 markers per cluster ===
Cluster 0: PLVAP, RAMP2, VWF, SPARCL1, IGFBP7
Cluster 1: SRP14, RAMP2, CAV1, A2M, CLDN5
Cluster 2: COL1A2, DCN, COL1A1, CTSK, C1S
Cluster 3: MYL9, IGFBP7, CALD1, TPM2, TAGLN
Cluster 4: KRT14, KRT17, KRT5, TAGLN, MT1X
Cluster 5: MS4A1, HLA-DRA, CD37, CD74, CD79A
Cluster 6: STMN1, HMGB2, TUBB, HMGN2, TUBA1B
Cluster 7: MZB1, SSR4, SEC11C, DERL3, FKBP11
Cluster 8: STMN1, UBE2C, BIRC5, CKS1B, HMGB1
Cluster 9: CCL5, NKG7, CCL4, CD8A, CD3E
Cluster 10: NKG7, GNLY, KLRD1, CTSW, CCL5
Cluster 11: B2M, IL32, CD2, CD3D, PTPRC
Cluster 12: IL7R, RPS27, RPL30, RPS25, BTG1
Cluster 13: TACSTD2, KRT7, SLPI, KRT15, MGST1
Cluster 14: TYROBP, FTL, AIF1, FCER1G, HLA

In [6]:
# ----------------------------
# Cell 6 — Canonical marker validation
# Confirms cluster annotations using known lineage markers
# All values from adata.raw (full gene, log-normalised, unscaled)
# ----------------------------
adata1_raw = adata1.raw.to_adata()
adata1_raw.obs["leiden_0.6"] = adata1.obs["leiden_0.6"].values

canonical_markers = {
    "T cells (CD3D, CD3E, TCF7)": ["CD3D", "CD3E", "TCF7", "IL7R", "CCR7"],
    "NK/Cytotoxic (NKG7, GNLY, PRF1)": ["NKG7", "GNLY", "PRF1", "KLRD1", "GZMB"],
    "Activated T (FOS, JUN, CD69)": ["CD69", "FOS", "JUN", "NFKBIA", "DUSP1"],
    "Macrophages (CD68, LYZ, TYROBP)": ["CD68", "LYZ", "TYROBP", "CD163"],
    "B cells (CD79A, MS4A1, CD19)": ["CD79A", "MS4A1", "CD19"],
    "Monocytes/DC (S100A8, CD14)": ["S100A8", "S100A9", "CD14", "FCGR3A", "VCAN"]
}

print("=== Canonical marker expression per cluster ===\n")
for pop, genes in canonical_markers.items():
    print(f"{pop}:")
    for gene in genes:
        if gene in adata1_raw.var_names:
            vals = []
            for cl in sorted(adata1.obs["leiden_0.6"].unique(), key=int):
                mask = (adata1_raw.obs["leiden_0.6"] == cl).values
                X = adata1_raw[mask, gene].X
                if issparse(X): X = X.toarray()
                vals.append(f"C{cl}={X.mean():.2f}")
            print(f"  {gene}: {', '.join(vals)}")
    print()

del adata1_raw
gc.collect()

=== Canonical marker expression per cluster ===

T cells (CD3D, CD3E, TCF7):
  CD3D: C0=0.80, C1=0.68, C2=0.78, C3=0.51, C4=0.14, C5=0.19
  CD3E: C0=0.74, C1=0.60, C2=0.70, C3=0.47, C4=0.11, C5=0.17
  TCF7: C0=1.09, C1=0.65, C2=0.77, C3=0.57, C4=0.27, C5=0.18
  IL7R: C0=1.48, C1=1.30, C2=1.27, C3=0.91, C4=0.24, C5=0.45
  CCR7: C0=0.49, C1=0.34, C2=0.36, C3=0.27, C4=0.42, C5=0.09

NK/Cytotoxic (NKG7, GNLY, PRF1):
  NKG7: C0=0.40, C1=0.58, C2=1.22, C3=0.55, C4=0.06, C5=0.24
  GNLY: C0=0.42, C1=0.58, C2=1.32, C3=0.66, C4=0.12, C5=0.31
  PRF1: C0=0.40, C1=0.54, C2=1.13, C3=0.58, C4=0.10, C5=0.27
  KLRD1: C0=0.30, C1=0.49, C2=0.81, C3=0.38, C4=0.07, C5=0.23
  GZMB: C0=0.15, C1=0.31, C2=0.63, C3=0.27, C4=0.06, C5=0.13

Activated T (FOS, JUN, CD69):
  CD69: C0=0.91, C1=1.45, C2=0.97, C3=0.79, C4=0.76, C5=0.57
  FOS: C0=0.89, C1=1.89, C2=0.88, C3=1.49, C4=0.60, C5=2.69
  JUN: C0=0.65, C1=1.43, C2=0.64, C3=1.01, C4=0.75, C5=1.51
  NFKBIA: C0=0.87, C1=1.43, C2=0.92, C3=1.06, C4=0.69, C5=1.49
  D

9895

In [7]:
# ----------------------------
# Cell 7 — Cell type annotation
# GSE114725: resolution 0.6, 6 clusters
# All annotations validated by canonical markers (Cell 6) and CellTypist (Cell 8)
# ----------------------------
cluster_labels_1 = {
    "0": "T cells",
    "1": "Activated T cells",
    "2": "NK/Cytotoxic T cells",
    "3": "Macrophages",
    "4": "B cells",
    "5": "Monocytes/DC"
}

# GSE176078: resolution 0.6, 26 clusters
cluster_labels_2 = {
    "0": "Endothelial cells", "1": "Endothelial cells",
    "2": "CAFs", "3": "PVL", "4": "Basal epithelial",
    "5": "B cells", "6": "Cycling cells", "7": "Plasma cells",
    "8": "Cycling epithelial", "9": "CD8 T cells",
    "10": "NK cells", "11": "T cells", "12": "Naive/memory T cells",
    "13": "Luminal epithelial", "14": "Macrophages",
    "15": "Monocytes/DC", "16": "Cycling myeloid", "17": "pDC",
    "18": "Luminal epithelial", "19": "Luminal epithelial",
    "20": "Epithelial", "21": "Epithelial",
    "22": "Luminal epithelial", "23": "Luminal epithelial",
    "24": "Luminal epithelial", "25": "Luminal epithelial"
}

adata1.obs["cell_type"] = adata1.obs["leiden_0.6"].map(cluster_labels_1)
adata2.obs["cell_type"] = adata2.obs["leiden_0.6"].map(cluster_labels_2)

print("GSE114725 cell types:")
print(adata1.obs["cell_type"].value_counts())
print("\nGSE176078 cell types (immune only shown):")
immune = ["T cells", "CD8 T cells", "NK cells", "Naive/memory T cells",
          "B cells", "Plasma cells", "Macrophages", "Monocytes/DC", "pDC"]
print(adata2.obs[adata2.obs["cell_type"].isin(immune)]["cell_type"].value_counts())

GSE114725 cell types:
cell_type
T cells                 16554
NK/Cytotoxic T cells     9811
Macrophages              8624
Activated T cells        5353
Monocytes/DC             3530
B cells                   790
Name: count, dtype: int64

GSE176078 cell types (immune only shown):
cell_type
Naive/memory T cells    11590
CD8 T cells              9387
Macrophages              8705
T cells                  5989
B cells                  2791
Plasma cells             2583
NK cells                 2438
pDC                       315
Monocytes/DC               22
Name: count, dtype: int64


In [9]:
# ----------------------------
# Cell 8 — CellTypist automated annotation
# Model: Immune_All_High.pkl
# Run on adata.raw (log-normalised unscaled full gene matrix)
# GSE114725: majority_voting=True
# GSE176078: batched (10,000 cells) due to memory constraints
# ----------------------------
import celltypist
from celltypist import models

models.download_models(force_update=False)
model = models.Model.load(model="Immune_All_High.pkl")

# GSE114725
adata1_raw_ct = adata1.raw.to_adata()
predictions1 = celltypist.annotate(
    adata1_raw_ct, model=model, majority_voting=True)
adata1.obs["celltypist"] = predictions1.predicted_labels["majority_voting"].values
del adata1_raw_ct
gc.collect()
print("GSE114725 CellTypist done")
print(adata1.obs["celltypist"].value_counts().head(10))

# GSE176078 — batched
batch_size = 10000
all_preds = []
for start in range(0, adata2.n_obs, batch_size):
    end = min(start + batch_size, adata2.n_obs)
    batch = adata2.raw.to_adata()[start:end]
    pred = celltypist.annotate(batch, model=model, majority_voting=False)
    all_preds.append(pred.predicted_labels["predicted_labels"])
    print(f"  Batch {start}-{end} done")
    gc.collect()

adata2.obs["celltypist"] = pd.concat(all_preds).values
print("\nGSE176078 CellTypist done")
print(adata2.obs["celltypist"].value_counts().head(10))

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\celltypist\classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv
📂 Storing models in C:\Users\annam\.celltypist\data\models
⏩ Skipping [1/61]: Immune_All_Low.pkl (file exists)
⏩ Skipping [2/61]: Immune_All_High.pkl (file exists)
⏩ Skipping [3/61]: Adult_COVID19_PBMC.pkl (file exists)
⏩ Skipping [4/61]: Adult_CynomolgusMacaque_Hippocampus.pkl (file exists)
⏩ Skipping [5/61]: Adult_Human_MTG.pkl (file exists)
⏩ Skipping [6/61]: Adult_Human_PancreaticIslet.pkl (file exists)
⏩ Skipping [7/61]: Adult_Human_PrefrontalCortex.pkl (file exists)
⏩ Skipping [8/61]: Adult_Human_Skin.pkl (file exists)
⏩ Skipping [9/61]: Adult_Human_Vascular.pkl (file exists)
⏩ Skipping [10/61]: Adult_Mouse_Gut.pkl (file exists)
⏩ Skipping [11/61]: Adult_Mouse_OlfactoryBulb.pkl (file exists)
⏩ Skipping [12/61]: Adult_Pig_Hippocampus.pkl (file exists)
⏩ Skipping [13/

GSE114725 CellTypist done
celltypist
T cells        36054
Macrophages     3107
ILC             1831
B cells         1797
Monocytes       1248
Mast cells       287
DC               257
pDC               81
Name: count, dtype: int64


🔬 Input data has 10000 cells and 27343 genes
🔗 Matching reference genes in the model
🧬 5340 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


  Batch 0-10000 done


🔬 Input data has 10000 cells and 27343 genes
🔗 Matching reference genes in the model
🧬 5340 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


...

  Batch 70000-80000 done


🔬 Input data has 10000 cells and 27343 genes
🔗 Matching reference genes in the model
🧬 5340 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


  Batch 80000-90000 done


🔬 Input data has 1425 cells and 27343 genes
🔗 Matching reference genes in the model
🧬 5340 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


  Batch 90000-91425 done

GSE176078 CellTypist done
celltypist
T cells              29490
Epithelial cells     23837
Macrophages           8758
Endothelial cells     8112
Fibroblasts           7592
ILC                   3859
B cells               3436
Plasma cells          2595
DC                    1645
Monocytes             1118
Name: count, dtype: int64


In [10]:
# ----------------------------
# Cell 9 — Annotated UMAPs
# ----------------------------
# GSE114725 — cell type + tissue + patient
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
sc.pl.umap(adata1, color="cell_type", title="GSE114725 — Cell Types",
           legend_loc="right margin", legend_fontsize=10,
           frameon=True, ax=axes[0], show=False)
sc.pl.umap(adata1, color="tissue", title="GSE114725 — Tissue",
           legend_loc="right margin", legend_fontsize=10,
           frameon=True, ax=axes[1], show=False)
sc.pl.umap(adata1, color="patient", title="GSE114725 — Patient",
           legend_loc="right margin", legend_fontsize=10,
           frameon=True, ax=axes[2], show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_annotated_umap_v2.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

# GSE176078 — cell type
fig, ax = plt.subplots(figsize=(12, 9))
sc.pl.umap(adata2, color="cell_type", title="GSE176078 — Cell Types",
           legend_loc="right margin", legend_fontsize=8,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE176078_annotated_umap_v2.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

# CellTypist UMAPs
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sc.pl.umap(adata1, color="celltypist", title="GSE114725 — CellTypist",
           legend_loc="right margin", legend_fontsize=8,
           frameon=True, ax=axes[0], show=False)
sc.pl.umap(adata2, color="celltypist", title="GSE176078 — CellTypist",
           legend_loc="right margin", legend_fontsize=7,
           frameon=True, ax=axes[1], show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "celltypist_umap_both_v2.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("All UMAPs saved")

... storing 'cell_type' as categorical
... storing 'celltypist' as categorical


All UMAPs saved


In [11]:
# ----------------------------
# Cell 10 — Save annotated objects and annotation tables
# ----------------------------
adata1.write(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad", compression="gzip")
adata2.write(PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad", compression="gzip")

adata1.obs[["leiden_0.6", "cell_type"]].drop_duplicates().sort_values(
    "leiden_0.6").to_csv(RESULTS_DIR / "GSE114725_cluster_annotations_v2.csv")
adata2.obs[["leiden_0.6", "cell_type"]].drop_duplicates().sort_values(
    "leiden_0.6").to_csv(RESULTS_DIR / "GSE176078_cluster_annotations_v2.csv")

print("Saved:")
print(f"  GSE114725: {adata1.n_obs} cells, {adata1.obs['cell_type'].nunique()} cell types")
print(f"  GSE176078: {adata2.n_obs} cells, {adata2.obs['cell_type'].nunique()} cell types")

Saved:
  GSE114725: 44662 cells, 6 cell types
  GSE176078: 91425 cells, 18 cell types


In [12]:
# ----------------------------
# Cell 11 — Stretch 1: T cell sub-clustering (GSE114725)
# Extract all T cell populations and re-cluster independently
# Gives finer resolution on T cell states than broad clustering
# ----------------------------
t_cell_labels = ["T cells", "Activated T cells", "NK/Cytotoxic T cells"]
adata1_tcells = adata1[adata1.obs["cell_type"].isin(t_cell_labels)].copy()

print(f"T cells extracted: {adata1_tcells.n_obs} cells")
print(adata1_tcells.obs["cell_type"].value_counts())

sc.pp.neighbors(adata1_tcells, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30)
sc.tl.umap(adata1_tcells, random_state=42)

for res in [0.3, 0.5, 0.7]:
    sc.tl.leiden(adata1_tcells, resolution=res,
                 key_added=f"tcell_leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False)
    print(f"Resolution {res}: {adata1_tcells.obs[f'tcell_leiden_{res}'].nunique()} clusters")

T cells extracted: 31718 cells
cell_type
T cells                 16554
NK/Cytotoxic T cells     9811
Activated T cells        5353
Name: count, dtype: int64
Resolution 0.3: 2 clusters
Resolution 0.5: 3 clusters
Resolution 0.7: 5 clusters


In [13]:
# Cell 12 — T cell marker genes and annotation
sc.tl.rank_genes_groups(adata1_tcells, groupby="tcell_leiden_0.7",
                        method="wilcoxon", key_added="rank_genes_tcell")

tcell_markers = sc.get.rank_genes_groups_df(
    adata1_tcells, group=None, key="rank_genes_tcell")

print("T cell sub-cluster markers:")
for cl in sorted(tcell_markers["group"].unique(), key=lambda x: int(x)):
    genes = tcell_markers[tcell_markers["group"] == cl].head(8)["names"].tolist()
    print(f"  Cluster {cl}: {', '.join(genes)}")

tcell_labels = {
    "0": "NK/Cytotoxic T cells",
    "1": "Resting T cells",
    "2": "Naive/Memory T cells",
    "3": "Activated T cells",
    "4": "B cell contamination"
}

adata1_tcells.obs["tcell_subtype"] = adata1_tcells.obs["tcell_leiden_0.7"].map(tcell_labels)

print("\nT cell subtype counts:")
print(adata1_tcells.obs["tcell_subtype"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.umap(adata1_tcells, color="tcell_subtype",
           title="GSE114725 — T Cell Sub-clusters",
           legend_loc="right margin", legend_fontsize=10,
           frameon=True, ax=axes[0], show=False)
sc.pl.umap(adata1_tcells, color="cell_type",
           title="Original broad annotation",
           legend_loc="right margin", legend_fontsize=10,
           frameon=True, ax=axes[1], show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_tcell_annotated_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

tcell_markers.groupby("group").head(10).to_csv(
    RESULTS_DIR / "GSE114725_tcell_subcluster_markers.csv", index=False)

adata1_tcells.write(
    PROCESSED_DIR / "GSE114725_tcells_subclustered.h5ad", compression="gzip")
print("Saved T cell sub-clustered object")

T cell sub-cluster markers:
  Cluster 0: RPLP2, RPL13, TPT1, RPL13A, RPS12, RPL30, RPS8, RPS29
  Cluster 1: RPS27, TPT1, RPLP2, RPL13, RPL30, RPS29, RPS28, RPS15A
  Cluster 2: DUSP1, FOS, CD74, SRGN, JUN, ZFP36, HLA-DRA, FOSB
  Cluster 3: CPA3, TPSB2, KIT, MS4A2, TPSAB1, VWA5A, GATA2, SLC18A2
  Cluster 4: NKG7, GNLY, PRF1, B2M, CCL5, HLA-B, CST7, FGFBP2

T cell subtype counts:
tcell_subtype
NK/Cytotoxic T cells    10057
Naive/Memory T cells     7953
Resting T cells          7307
B cell contamination     6200
Activated T cells         201
Name: count, dtype: int64
Saved T cell sub-clustered object


In [14]:
# ----------------------------
# Cell 13 — Stretch 2: Macrophage sub-clustering (GSE114725)
# Extract macrophages and re-cluster independently
# Motivated by: most DEGs in broad cluster DE, highest proportion in TNBC
# ----------------------------
adata1_mac = adata1[adata1.obs["cell_type"] == "Macrophages"].copy()

print(f"Macrophages extracted: {adata1_mac.n_obs} cells")
print(adata1_mac.obs["tissue"].value_counts())

sc.pp.neighbors(adata1_mac, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30)
sc.tl.umap(adata1_mac, random_state=42)

for res in [0.3, 0.5, 0.7]:
    sc.tl.leiden(adata1_mac, resolution=res,
                 key_added=f"mac_leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False)
    print(f"Resolution {res}: {adata1_mac.obs[f'mac_leiden_{res}'].nunique()} clusters")

Macrophages extracted: 8624 cells
tissue
TUMOR        4598
BLOOD        2580
LYMPHNODE     821
NORMAL        625
Name: count, dtype: int64
Resolution 0.3: 2 clusters
Resolution 0.5: 3 clusters
Resolution 0.7: 4 clusters


In [15]:
# Cell 14 — Macrophage marker genes and annotation
sc.tl.rank_genes_groups(adata1_mac, groupby="mac_leiden_0.7",
                        method="wilcoxon", key_added="rank_genes_mac")

mac_markers = sc.get.rank_genes_groups_df(
    adata1_mac, group=None, key="rank_genes_mac")

print("Macrophage sub-cluster markers:")
for cl in sorted(mac_markers["group"].unique(), key=lambda x: int(x)):
    genes = mac_markers[mac_markers["group"] == cl].head(8)["names"].tolist()
    print(f"  Cluster {cl}: {', '.join(genes)}")

if "n_genes_by_counts" in adata1_mac.obs.columns:
    median_per_cluster = adata1_mac.obs.groupby(
        "mac_leiden_0.7")["n_genes_by_counts"].median()
    overall_median = median_per_cluster.median()
    cluster4_label = "Low quality" if median_per_cluster.get(
        "4", overall_median) < 0.7 * overall_median else "Resting/Resident"
    print(f"\nCluster 4 label: {cluster4_label}")
else:
    cluster4_label = "Resting/Resident"

mac_labels = {
    "0": "LAM-like (C1Q+/APOE+)",
    "1": "Antigen-presenting (HLA-DR+)",
    "2": "Monocyte-like (S100A8/A9+)",
    "3": "NK/T cell contamination",
    "4": cluster4_label,
    "5": "B cell contamination"
}

adata1_mac.obs["mac_subtype"] = adata1_mac.obs["mac_leiden_0.7"].map(mac_labels)

print("\nMacrophage subtype counts:")
print(adata1_mac.obs["mac_subtype"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.umap(adata1_mac, color="mac_subtype",
           title="GSE114725 — Macrophage Sub-clusters",
           legend_loc="right margin", legend_fontsize=9,
           frameon=True, ax=axes[0], show=False)
sc.pl.umap(adata1_mac, color="tissue",
           title="Tissue origin",
           legend_loc="right margin", legend_fontsize=9,
           frameon=True, ax=axes[1], show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_macrophage_subclusters_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

mac_markers.groupby("group").head(10).to_csv(
    RESULTS_DIR / "GSE114725_macrophage_subcluster_markers.csv", index=False)

adata1_mac.write(
    PROCESSED_DIR / "GSE114725_macrophages_subclustered.h5ad", compression="gzip")
print("Saved macrophage sub-clustered object")

Macrophage sub-cluster markers:
  Cluster 0: RPLP2, RPS27, RPL13A, RPL13, RPS8, RPS12, RPS29, RPLP1
  Cluster 1: CST3, HLA-DRA, CD74, HLA-DRB1, CTSB, GPX1, HLA-DPA1, NPC2
  Cluster 2: B2M, PRF1, NKG7, GNLY, HLA-A, HLA-B, GZMA, CSTB
  Cluster 3: IFITM2, S100A8, S100A9, S100A6, SRGN, SH3BGRL3, NEAT1, FPR1

Cluster 4 label: Resting/Resident

Macrophage subtype counts:
mac_subtype
Antigen-presenting (HLA-DR+)    2964
LAM-like (C1Q+/APOE+)           2020
Monocyte-like (S100A8/A9+)      1953
NK/T cell contamination         1687
Name: count, dtype: int64
Saved macrophage sub-clustered object


In [16]:
# ----------------------------
# Cell 15 — Stretch 3: Cell type proportions
# GSE114725: per patient
# GSE176078: per clinical subtype (ER+/HER2+/TNBC)
# ----------------------------

# GSE114725 — per patient
props1 = adata1.obs.groupby(
    ["patient", "cell_type"]).size().unstack(fill_value=0)
props1_pct = props1.div(props1.sum(axis=1), axis=0) * 100
props1_pct.to_csv(RESULTS_DIR / "GSE114725_celltype_proportions.csv")

fig, ax = plt.subplots(figsize=(12, 6))
props1_pct.T.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_xlabel("Patient")
ax.set_ylabel("Proportion (%)")
ax.set_title("GSE114725 — Cell Type Proportions per Patient")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_celltype_proportions.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

# GSE176078 — per subtype
props2 = adata2.obs.groupby(
    ["subtype", "cell_type"]).size().unstack(fill_value=0)
props2_pct = props2.div(props2.sum(axis=1), axis=0) * 100
props2_pct.to_csv(RESULTS_DIR / "GSE176078_celltype_proportions_by_subtype.csv")

fig, ax = plt.subplots(figsize=(12, 7))
props2_pct.T.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_xlabel("Subtype")
ax.set_ylabel("Proportion (%)")
ax.set_title("GSE176078 — Cell Type Proportions per Breast Cancer Subtype")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE176078_celltype_proportions_subtype.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

print("GSE114725 proportions:")
print(props1_pct.round(1))
print("\nGSE176078 proportions:")
print(props2_pct.round(1))

GSE114725 proportions:
cell_type  T cells  Activated T cells  NK/Cytotoxic T cells  Macrophages  \
patient                                                                    
BC1           41.5               10.5                  25.6         15.4   
BC2           41.4               12.2                  23.9         16.5   
BC3           18.8               14.3                   9.3         33.4   
BC4           41.5               10.4                  24.5         16.8   
BC5           25.3               18.6                  14.1         26.1   
BC6           18.6               13.4                  12.5         32.4   
BC7           28.4               17.0                  15.0         24.8   
BC8           25.2               16.3                  15.3         25.9   

cell_type  B cells  Monocytes/DC  
patient                           
BC1            1.4           5.6  
BC2            2.4           3.6  
BC3            0.6          23.8  
BC4            2.1           4.9  
BC5   

In [17]:
# ----------------------------
# Cell 16 — Stretch 4: Reference atlas comparison (GSE176078)
# Cross-tabulate our annotations against Wu et al. 2021 published labels
# Only possible for GSE176078 — GSE114725 has no published per-cell annotations
# ----------------------------
comparison = pd.crosstab(
    adata2.obs["cell_type"],
    adata2.obs["celltype_major"],
    normalize="index"
).round(3) * 100

comparison.to_csv(RESULTS_DIR / "GSE176078_our_vs_published_comparison.csv")

print("Our annotation vs Wu et al. 2021 (% of our cells in each published category):")
print(comparison)

print("\nTop concordance per cell type:")
for ct in comparison.index:
    top = comparison.loc[ct].idxmax()
    pct = comparison.loc[ct].max()
    print(f"  {ct} -> {top}: {pct:.1f}%")

Our annotation vs Wu et al. 2021 (% of our cells in each published category):
celltype_major        B-cells  CAFs  Cancer Epithelial  Endothelial  Myeloid  \
cell_type                                                                      
B cells                  96.9   0.0                0.3          0.8      1.0   
Basal epithelial          0.0   0.1                4.4          0.1      0.0   
CAFs                      0.0  89.3               10.3          0.0      0.0   
CD8 T cells               0.0   0.0                0.0          0.0      0.0   
Cycling cells             0.0   0.0                2.7          0.0      0.5   
Cycling epithelial        0.0   0.0               98.5          0.0      0.0   
Cycling myeloid           0.0   0.0              100.0          0.0      0.0   
Endothelial cells         0.0   0.0                0.0         99.5      0.1   
Epithelial                0.1   0.0               97.3          0.0      0.0   
Luminal epithelial        0.0   0.0       